### **Notebook 3: Análisis de identificadores (pacientes, hospitales y médicos).**

##### **Objetivo:** Este notebook busca evaluar el rol de las variables identificadoras (`CIP_ENCRIPTADO`, `COD_HOSPITAL`, `MEDICOINTERV1` y `MEDICOALTA`) dentro del modelado predictivo para determinar su viabilidad como características de entrada en los algoritmos de Machine Learning.

##### **Procedimiento:**
1. **Análisis de pacientes repetidos**: Cuantifica la frecuencia de episodios por paciente único a lo largo de los años para evaluar la necesidad de agrupar o descartar el ID.
2. **Análisis de distribución de hospitales**: Analiza la concentración de episodios por hospital (`COD_HOSPITAL`) para evaluar su cardinalidad e impacto territorial.
3. **Análisis de médicos intervinientes y de alta**: Evalúa la viabilidad de mantener la trazabilidad de los profesionales tratantes, observando su cardinalidad.

In [1]:
import pandas as pd # Librería para manipulación de datos tabulares.
import os # Librería para manejo de rutas y directorios.
import warnings # Librería para control de advertencias.
# Suprime advertencias de tipos mixtos durante la carga de archivos grandes.
warnings.simplefilter(action='ignore', category=pd.errors.DtypeWarning)

# 1. Configuración de rutas y archivos
# Carpeta que contiene los datasets GRD previamente clasificados.
carpeta = "../../Datos/Datos clasificados"
# Carpeta donde se almacenarán los resultados del análisis.
ruta_resultados = "../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos"
# Crea la carpeta de resultados si no existe.
os.makedirs(ruta_resultados, exist_ok=True)
# Lista de archivos anuales que serán procesados.
archivos = [ # Lista de archivos clasificados en el notebook anterior.
    "GRD_CLASIFICADO_2019.csv",
    "GRD_CLASIFICADO_2020.csv",
    "GRD_CLASIFICADO_2021.csv",
    "GRD_CLASIFICADO_2022.csv",
    "GRD_CLASIFICADO_2023.csv",
    "GRD_CLASIFICADO_2024.csv"
]

##### **Identificador de pacientes repetidos: `CIP_ENCRIPTADO`**

1. **`CIP_ENCRIPTADO`**
- **Tipo de variable:** Demográfica (identificador del paciente).
- **Descripción:** Código identificador único (encriptado) de cada paciente que ingresa al sistema hospitalario.
- **Veredicto:** Se descartó su uso en la fase de modelado, ya que el identificador encriptado de cada paciente no presenta una relación clínica ni operativa con su nivel de criticidad. Además, se identificaron pacientes con volúmenes de ingresos sospechosamente elevados o humanamente imposibles, como 8.307 episodios en 2019 asociados al registro `SIN INFORMACIÓN` o 523 episodios para el identificador `95162030` en 2022. Esto sugiere que la variable se encuentra probablemente contaminada con identificadores institucionales o códigos por defecto utilizados para pacientes sin registro. En consecuencia, la aplicación de transformaciones como `One-Hot Encoding` sobre una variable con miles de categorías generaría un conjunto de datos altamente dimensional y propenso al sobreajuste (`overfitting`), afectando negativamente el rendimiento de los algoritmos de selección de características. Asimismo, cada episodio será evaluado como una instancia independiente según su contexto clínico.

In [2]:
# 1. Análisis de pacientes repetidos (CIP_ENCRIPTADO)
# Diccionario para almacenar los 5 pacientes con más episodios por año.
top5_repetidos_por_año = {}
# Recorre todos los datasets anuales.
for archivo in archivos:
    # Construye la ruta completa del archivo.
    ruta = os.path.join(carpeta, archivo)
    # Extrae el año desde el nombre del archivo.
    año = archivo[-8:-4]
    # Carga el dataset completo.
    df = pd.read_csv(ruta, low_memory=False)
    # Verifica la existencia del identificador anonimizado del paciente.
    if "CIP_ENCRIPTADO" in df.columns:
        # Convierte el identificador a texto para evitar inconsistencias.
        df["CIP_ENCRIPTADO"] = df["CIP_ENCRIPTADO"].astype(str)
        # Elimina registros con identificadores desconocidos o faltantes.
        df = df[
            ~df["CIP_ENCRIPTADO"].isin(
                ["SIN INFORMACION", "DESCONOCIDO"]
            )
        ]
        # Cuenta la cantidad de episodios asociados a cada paciente.
        conteo = df["CIP_ENCRIPTADO"].value_counts()
        # Conserva únicamente pacientes con más de un episodio.
        repetidos = conteo[conteo > 1]
        # Convierte el resultado a DataFrame.
        repetidos_df = repetidos.reset_index()
        # Renombra columnas para facilitar interpretación.
        repetidos_df.columns = ["CIP_ENCRIPTADO", "count"]
        # Define la ruta de salida del resultado anual.
        ruta_salida = f"{ruta_resultados}/repetidos_{año}.csv"
        # Exporta la tabla de pacientes repetidos.
        repetidos_df.to_csv(ruta_salida, index=False)
        # Guarda los cinco pacientes con mayor número de episodios.
        top5_repetidos_por_año[año] = repetidos_df.head(5)
        # Reporta estadísticas resumidas del año procesado.
        print(
            f"- {año}: {len(repetidos_df)} pacientes con múltiples episodios. "
            f"Max: {repetidos.max()} episodios. Guardado en {ruta_salida}"
        )

- 2019: 165128 pacientes con múltiples episodios. Max: 8307 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2019.csv
- 2020: 98379 pacientes con múltiples episodios. Max: 1893 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2020.csv
- 2021: 108690 pacientes con múltiples episodios. Max: 2044 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2021.csv
- 2022: 129107 pacientes con múltiples episodios. Max: 523 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2022.csv
- 2023: 151694 pacientes con múltiples episodios. Max: 341 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2023.csv
- 2024: 161533 pacientes con múltiples episodios. Max: 140 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2024.csv


In [5]:
# Pacientes repetidos entre 2019 y 2021
print("="*70)
print("TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2019-2021)")
print("="*70)
for año in ["2019", "2020", "2021"]: # Muestro la cantidad de pacientes repetidos y el máximo de episodios por año.
    print(f"\n- Año {año} (Máximo de episodios en {año}: {top5_repetidos_por_año[año]['count'].iloc[0]}):")
    print(top5_repetidos_por_año[año].to_string(index=False))

TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2019-2021)

- Año 2019 (Máximo de episodios en 2019: 8307):
 CIP_ENCRIPTADO  count
SIN INFORMACIÓN   8307
         608828    321
         317005    175
         363057    157
         284899    155

- Año 2020 (Máximo de episodios en 2020: 1893):
 CIP_ENCRIPTADO  count
SIN INFORMACIÓN   1893
        1322640    445
         608828    362
        1099683    279
          24929     79

- Año 2021 (Máximo de episodios en 2021: 2044):
CIP_ENCRIPTADO  count
           nan   2044
    95162030.0    294
    78492052.0    273
    96754933.0    103
    77066145.0     73


In [6]:
# Pacientes repetidos entre 2022 y 2024
print("="*70)
print("TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2022-2024)")
print("="*70)
for año in ["2022", "2023", "2024"]: # Muestro la cantidad de pacientes repetidos y el máximo de episodios por año.
    print(f"\n- Año {año} (Máximo de episodios en {año}: {top5_repetidos_por_año[año]['count'].iloc[0]}):")
    print(top5_repetidos_por_año[año].to_string(index=False))

TOP 5 PACIENTES CON MÁS EPISODIOS HOSPITALARIOS (2022-2024)

- Año 2022 (Máximo de episodios en 2022: 523):
CIP_ENCRIPTADO  count
      95162030    523
      78492052    302
      96754933    272
      98437046    125
      97796953     83

- Año 2023 (Máximo de episodios en 2023: 341):
CIP_ENCRIPTADO  count
      95162030    341
      78492052    334
      95701049     67
      73381193     55
      78875009     54

- Año 2024 (Máximo de episodios en 2024: 140):
CIP_ENCRIPTADO  count
      78492052    140
      92276704     99
      77008364     79
      95162030     70
      78710945     63


In [7]:
# Genero análisis de tendencias temporal
print("\n" + "="*70)
print("ANÁLISIS DE TENDENCIAS TEMPORALES (Mayor repetición por año)")
print("="*70)
for año in ["2019", "2020", "2021", "2022", "2023", "2024"]: # Indico el paciente con mayor repetición por año.
    max_rep = top5_repetidos_por_año[año]['count'].iloc[0]
    print(f"  {año}: {max_rep:3d} episodios (paciente con mayor repetición)")


ANÁLISIS DE TENDENCIAS TEMPORALES (Mayor repetición por año)
  2019: 8307 episodios (paciente con mayor repetición)
  2020: 1893 episodios (paciente con mayor repetición)
  2021: 2044 episodios (paciente con mayor repetición)
  2022: 523 episodios (paciente con mayor repetición)
  2023: 341 episodios (paciente con mayor repetición)
  2024: 140 episodios (paciente con mayor repetición)


In [ ]:
# Lista para almacenar todos los CIP_ENCRIPTADO de todos los años
todos_los_cip = []

# Recorre todos los datasets anuales
for archivo in archivos:
    # Construye la ruta completa del archivo
    ruta = os.path.join(carpeta, archivo)

    # Carga el dataset
    df = pd.read_csv(ruta, low_memory=False)

    # Verifica que exista la columna
    if "CIP_ENCRIPTADO" in df.columns:
        # Convierte a texto
        df["CIP_ENCRIPTADO"] = df["CIP_ENCRIPTADO"].astype(str)

        # Elimina registros inválidos
        df = df[
            ~df["CIP_ENCRIPTADO"].isin(
                ["SIN INFORMACION", "DESCONOCIDO"]
            )
        ]

        # Agrega todos los identificadores a la lista global
        todos_los_cip.extend(df["CIP_ENCRIPTADO"].tolist())

# Convierte la lista a una Serie
todos_los_cip = pd.Series(todos_los_cip)

# Cuenta episodios por paciente considerando todos los años
conteo_global = todos_los_cip.value_counts()

# Conserva solo pacientes con más de un episodio
repetidos_global = conteo_global[conteo_global > 1]

# Convierte a DataFrame
repetidos_global_df = repetidos_global.reset_index()
repetidos_global_df.columns = ["CIP_ENCRIPTADO", "count"]

# Muestra estadísticas
print(f"Pacientes únicos: {conteo_global.shape[0]:,}")
print(f"Pacientes con múltiples episodios: {repetidos_global.shape[0]:,}")
print(f"Máximo de episodios para un paciente: {repetidos_global.max():,}")

print("\nTop 5 pacientes con más episodios (2019-2024):")
print(repetidos_global_df.head(5))

Pacientes únicos: 4,214,737
Pacientes con múltiples episodios: 967,699
Máximo de episodios para un paciente: 10,200

Top 5 pacientes con más episodios (2019-2024):
    CIP_ENCRIPTADO  count
0  SIN INFORMACIÓN  10200
1              nan   2044
2         95162030    934
3         78492052    776
4           608828    683


##### **Distribución de hospitales en los episodios oncológicos (COD_HOSPITAL)**

2. `COD_HOSPITAL`
- **Tipo de variable:** Hospitalaria.
- **Descripción:** Código numérico que identifica al establecimiento de salud específico donde ocurrió el episodio y el egreso del paciente.
- **Veredicto:** Se **descartó** su uso en la fase de modelado debido a su elevada dimensionalidad (entre 65 y 72 categorías distintas por año). Además, esta variable se encuentra jerárquicamente anidada dentro de la variable `SERVICIO_SALUD`, la cual agrupa establecimientos según macrozonas geográficas y presenta una menor cardinalidad. Por lo tanto, mantener ambas variables podría introducir colinealidad en el modelo. Adicionalmente, el propósito principal de este estudio es apoyar la planificación sanitaria y la gestión de recursos a nivel macro, orientadas a organismos como FONASA y MINSAL. En este contexto, un análisis centrado en hospitales individuales excede el alcance epidemiológico de la caracterización propuesta. 

In [8]:
# 2. Análisis de distribución de hospitales (COD_HOSPITAL)
# Carpeta donde se almacenarán los resultados asociados a hospitales.
ruta_resultados_hospitales = "../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales"
# Crea la carpeta de salida si no existe.
os.makedirs(ruta_resultados_hospitales, exist_ok=True)
# Almacena los cinco hospitales con más episodios por año.
top5_hospitales_por_año = {}
# Almacena estadísticas resumidas para el reporte final.
resumen_hospitales = {}
# Recorre todos los datasets anuales.
for archivo in archivos:
    # Construye la ruta completa del archivo.
    ruta = os.path.join(carpeta, archivo)
    # Extrae el año desde el nombre del archivo.
    año = archivo[-8:-4]
    # Carga el dataset correspondiente.
    df = pd.read_csv(ruta, low_memory=False)
    # Verifica la existencia del código hospitalario.
    if "COD_HOSPITAL" in df.columns:
        # Convierte el código hospitalario a texto.
        df["COD_HOSPITAL"] = df["COD_HOSPITAL"].astype(str)
        # Cuenta episodios por hospital.
        conteo_hosp = df["COD_HOSPITAL"].value_counts()
        # Convierte el resultado a DataFrame.
        hospitales_df = conteo_hosp.reset_index()
        # Renombra columnas descriptivamente.
        hospitales_df.columns = ["COD_HOSPITAL", "count"]
        # Define el archivo de salida.
        ruta_salida_hosp = (
            f"{ruta_resultados_hospitales}/hospitales_{año}.csv"
        )
        # Exporta la distribución hospitalaria.
        hospitales_df.to_csv(ruta_salida_hosp, index=False)
        # Guarda los cinco hospitales con mayor volumen.
        top5_hospitales_por_año[año] = hospitales_df.head(5)
        # Calcula el número de hospitales distintos.
        num_hospitales = hospitales_df["COD_HOSPITAL"].nunique()
        # Calcula episodios acumulados en los cinco principales hospitales.
        top5_volume = hospitales_df.head(5)["count"].sum()
        # Calcula el total de episodios registrados.
        total_volume = hospitales_df["count"].sum()
        # Calcula el porcentaje de concentración de los cinco principales.
        concentracion = (top5_volume / total_volume) * 100
        # Guarda métricas resumidas para análisis posterior.
        resumen_hospitales[año] = (
            num_hospitales,
            concentracion
        )
        # Reporta resultados resumidos del año.
        print(
            f"- {año}: {num_hospitales} hospitales distintos. "
            f"Top 5 concentra {concentracion:.1f}% de episodios. "
            f"Guardado en {ruta_salida_hosp}"
        )

- 2019: 65 hospitales distintos. Top 5 concentra 18.1% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2019.csv
- 2020: 65 hospitales distintos. Top 5 concentra 18.0% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2020.csv
- 2021: 65 hospitales distintos. Top 5 concentra 18.6% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2021.csv
- 2022: 65 hospitales distintos. Top 5 concentra 17.8% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2022.csv
- 2023: 68 hospitales distintos. Top 5 concentra 17.0% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2023.csv
- 2024: 72 hospitales distintos. Top 5 concentra 16.5% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/

In [9]:
# Hospitales con más episodios entre 2019 y 2021
print("\n" + "="*70)
print("TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2019-2021)")
print("="*70)
for año in ["2019", "2020", "2021"]: # Muestro la cantidad de hospitales distintos, la concentración de los 5 principales y el máximo de episodios por año.
    max_hosp = top5_hospitales_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Hospital con mayor cantidad de episodios: {max_hosp}):")
    print(top5_hospitales_por_año[año].to_string(index=False))


TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2019-2021)

Año 2019 (Hospital con mayor cantidad de episodios: 55726):
COD_HOSPITAL  count
      114101  55726
      106100  42026
      118100  40134
      109100  35921
      121109  35091

Año 2020 (Hospital con mayor cantidad de episodios: 37040):
COD_HOSPITAL  count
      114101  37040
      113100  28652
      116105  25321
      109100  24903
      118100  24660

Año 2021 (Hospital con mayor cantidad de episodios: 44842):
COD_HOSPITAL  count
      114101  44842
      118100  28853
      113100  27955
      116105  25610
      109100  24947


In [10]:
# Hospitales con más episodios entre 2022 y 2024
print("\n" + "="*70)
print("TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2022-2024)")
print("="*70)
for año in ["2022", "2023", "2024"]:
    max_hosp = top5_hospitales_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Hospital con mayor cantidad de episodios: {max_hosp}):")
    print(top5_hospitales_por_año[año].to_string(index=False))


TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS (2022-2024)

Año 2022 (Hospital con mayor cantidad de episodios: 47426):
COD_HOSPITAL  count
      114101  47426
      118100  31167
      116105  30292
      120101  29101
      109100  28275

Año 2023 (Hospital con mayor cantidad de episodios: 49653):
COD_HOSPITAL  count
      114101  49653
      118100  34144
      116105  32148
      120101  31251
      121109  29786

Año 2024 (Hospital con mayor cantidad de episodios: 49411):
COD_HOSPITAL  count
      114101  49411
      118100  35148
      116105  33858
      120101  30527
      124105  30453


In [11]:
print("\nANÁLISIS DE TENDENCIAS TEMPORAL PARA HOSPITALES")
for año in ["2019", "2020", "2021", "2022", "2023", "2024"]:
    num_hosp, conc_pct = resumen_hospitales[año]
    print(f"  {año}: {num_hosp:2d} hospitales - Concentración (top 5): {conc_pct:5.1f}%")


ANÁLISIS DE TENDENCIAS TEMPORAL PARA HOSPITALES
  2019: 65 hospitales - Concentración (top 5):  18.1%
  2020: 65 hospitales - Concentración (top 5):  18.0%
  2021: 65 hospitales - Concentración (top 5):  18.6%
  2022: 65 hospitales - Concentración (top 5):  17.8%
  2023: 68 hospitales - Concentración (top 5):  17.0%
  2024: 72 hospitales - Concentración (top 5):  16.5%


In [3]:
# Análisis global de episodios por hospital (2019-2024)

# Lista para almacenar todos los códigos de hospital
todos_los_hospitales = []

# Recorre todos los datasets anuales
for archivo in archivos:
    # Construye la ruta completa del archivo
    ruta = os.path.join(carpeta, archivo)

    # Carga el dataset
    df = pd.read_csv(ruta, low_memory=False)

    # Verifica que exista la columna
    if "COD_HOSPITAL" in df.columns:
        # Agrega todos los códigos de hospital a la lista global
        todos_los_hospitales.extend(df["COD_HOSPITAL"].tolist())

# Convierte la lista a una Serie
todos_los_hospitales = pd.Series(todos_los_hospitales)

# Elimina valores faltantes (opcional, pero recomendable)
todos_los_hospitales = todos_los_hospitales.dropna()

# Cuenta episodios por hospital
conteo_hospitales = todos_los_hospitales.value_counts()

# Convierte a DataFrame
conteo_hospitales_df = conteo_hospitales.reset_index()
conteo_hospitales_df.columns = ["COD_HOSPITAL", "count"]

# Muestra estadísticas
print(f"Hospitales únicos: {conteo_hospitales.shape[0]:,}")
print(f"Total de episodios: {conteo_hospitales.sum():,}")

print("\nTop 10 hospitales con más episodios (2019-2024):")
print(conteo_hospitales_df.head(10))

Hospitales únicos: 72
Total de episodios: 5,808,455

Top 10 hospitales con más episodios (2019-2024):
   COD_HOSPITAL   count
0        114101  284098
1        118100  194106
2        116105  179092
3        113100  171323
4        109100  169699
5        120101  168941
6        121109  167101
7        124105  160095
8        115100  155037
9        110100  152295


##### **Distribución de médicos (a través de sus identificadores)**

3. **MEDICOINTERV1_ENCRIPTADO**
- **Tipo de variable:** Hospitalaria (identificador).
- **Descripción:** Código encriptado que identifica de forma única al médico responsable de realizar la primera intervención o procedimiento quirúrgico.
- **Veredicto:** Se **descartó** su uso en la fase de modelado, ya que registra hasta 11.164 valores únicos por año, lo que generaría una matriz altamente dispersa y penalizaría el desempeño de los algoritmos de selección de características. Por esta razón, su utilización resulta metodológicamente inviable. Además, el objetivo del estudio es caracterizar aspectos clínicos y asistenciales, tales como los procedimientos realizados, y no identificar a los profesionales que los ejecutaron. 

In [12]:
# 3. Análisis de Médicos Intervinientes (MEDICOINTERV1_ENCRIPTADO)
# Carpeta de resultados para médicos intervinientes.
ruta_resultados_med_interv = "../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención"
# Crea la carpeta de salida si no existe.
os.makedirs(ruta_resultados_med_interv, exist_ok=True)
# Almacena el top 5 de médicos intervinientes por año.
top5_med_interv_por_año = {}
# Recorre todos los datasets anuales.
for archivo in archivos:
    # Construye la ruta completa del archivo.
    ruta = os.path.join(carpeta, archivo)
    # Extrae el año desde el nombre del archivo.
    año = archivo[-8:-4]
    # Carga el dataset correspondiente.
    df = pd.read_csv(ruta, low_memory=False)
    # Verifica la existencia del identificador del médico interviniente.
    if "MEDICOINTERV1_ENCRIPTADO" in df.columns:
        # Convierte el identificador a texto.
        df["MEDICOINTERV1_ENCRIPTADO"] = (
            df["MEDICOINTERV1_ENCRIPTADO"].astype(str)
        )
        # Elimina registros sin información válida.
        df = df[
            ~df["MEDICOINTERV1_ENCRIPTADO"].isin(
                ["SIN INFORMACION", "DESCONOCIDO", "nan", "None"]
            )
        ]
        # Cuenta episodios asociados a cada médico.
        conteo_med = (
            df["MEDICOINTERV1_ENCRIPTADO"]
            .value_counts()
        )
        # Convierte el resultado a DataFrame.
        med_df = conteo_med.reset_index()
        # Renombra columnas descriptivamente.
        med_df.columns = [
            "MEDICOINTERV1_ENCRIPTADO",
            "count"
        ]
        # Define la ruta de salida.
        ruta_salida = (
            f"{ruta_resultados_med_interv}/"
            f"medicos_intervencion_{año}.csv"
        )
        # Exporta la distribución de médicos intervinientes.
        med_df.to_csv(
            ruta_salida,
            index=False,
            encoding="utf-8-sig"
        )
        # Guarda los cinco médicos con mayor volumen.
        top5_med_interv_por_año[año] = med_df.head(5)
        # Calcula el número de médicos distintos.
        num_med = med_df["MEDICOINTERV1_ENCRIPTADO"].nunique()
        # Calcula volumen acumulado del top 5.
        top5_volume = med_df.head(5)["count"].sum()
        # Calcula el total de episodios asociados.
        total_volume = med_df["count"].sum()
        # Calcula el porcentaje de concentración.
        concentracion = (top5_volume / total_volume) * 100
        # Reporta resultados resumidos.
        print(
            f"- {año}: {num_med} médicos intervinientes distintos. "
            f"Top 5 concentra {concentracion:.1f}% de episodios."
        )

- 2019: 9617 médicos intervinientes distintos. Top 5 concentra 9.0% de episodios.
- 2020: 10388 médicos intervinientes distintos. Top 5 concentra 8.5% de episodios.
- 2021: 9476 médicos intervinientes distintos. Top 5 concentra 7.4% de episodios.
- 2022: 10198 médicos intervinientes distintos. Top 5 concentra 7.0% de episodios.
- 2023: 10569 médicos intervinientes distintos. Top 5 concentra 4.8% de episodios.
- 2024: 11164 médicos intervinientes distintos. Top 5 concentra 4.5% de episodios.


In [13]:
for año in ["2019", "2020", "2021"]: # Médicos intervinientes más frecuentes entre 2019 y 2021
    max_hosp = top5_med_interv_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_interv_por_año[año].to_string(index=False))


Año 2019 (Médico con mayor cantidad de intervenciones: 29905 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
                 11769.0  29905
                  2657.0   9568
                  9016.0   6245
                    54.0   5961
                 11780.0   2514

Año 2020 (Médico con mayor cantidad de intervenciones: 19202 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
                 11780.0  19202
                 11769.0   9290
                  2657.0   3601
                  9016.0   1844
                  6667.0   1077

Año 2021 (Médico con mayor cantidad de intervenciones: 28198 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  28198
              68823337.0   1093
              68804909.0   1004
              70326745.0    947
              77772744.0    928


In [14]:
for año in ["2022", "2023", "2024"]: # Médicos intervinientes más frecuentes entre 2022 y 2024  
    max_hosp = top5_med_interv_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_interv_por_año[año].to_string(index=False))


Año 2022 (Médico con mayor cantidad de intervenciones: 32947 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  32947
              71043147.0   1230
              70326745.0   1150
              70517922.0   1149
              77772744.0   1053

Año 2023 (Médico con mayor cantidad de intervenciones: 23200 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  23200
              70517922.0   1598
              69957946.0   1491
              80800383.0   1177
              80847393.0   1150

Año 2024 (Médico con mayor cantidad de intervenciones: 22221 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0  22221
              71043147.0   1821
              70517922.0   1619
              68804909.0   1155
              71057175.0   1118


In [4]:
# Análisis global de médico de intervención (2019-2024)

# Lista para almacenar todos los códigos de médico de intervención 
medicos = []

# Recorre todos los datasets anuales
for archivo in archivos:
    # Construye la ruta completa del archivo
    ruta = os.path.join(carpeta, archivo)

    # Carga el dataset
    df = pd.read_csv(ruta, low_memory=False)

    # Verifica que exista la columna
    if "MEDICOINTERV1_ENCRIPTADO" in df.columns:
        # Agrega todos los códigos de médico de intervención a la lista global
        medicos.extend(df["MEDICOINTERV1_ENCRIPTADO"].tolist())

# Convierte la lista a una Serie
medicos = pd.Series(medicos)

# Elimina valores faltantes (opcional, pero recomendable)
medicos = medicos.dropna()

# Cuenta episodios por médico de intervención
conteo_medicos = medicos.value_counts()

# Convierte a DataFrame
conteo_medicos_df = conteo_medicos.reset_index()
conteo_medicos_df.columns = ["MEDICOINTERV1_ENCRIPTADO", "count"]

# Muestra estadísticas
print(f"Médicos de intervención únicos: {conteo_medicos.shape[0]:,}")
print(f"Total de episodios: {conteo_medicos.sum():,}")

print("\nTop 10 médicos de intervención con más episodios (2019-2024):")
print(conteo_medicos_df.head(10))

Médicos de intervención únicos: 28,752
Total de episodios: 3,200,752

Top 10 médicos de intervención con más episodios (2019-2024):
   MEDICOINTERV1_ENCRIPTADO   count
0                98438434.0  106566
1                   11769.0   39195
2                   11780.0   21716
3                    2657.0   13169
4                    9016.0    8089
5                      54.0    6983
6                70517922.0    5081
7                71043147.0    4844
8                77772744.0    4111
9                68804909.0    4006


4. **MEDICOALTA_ENCRIPTADO**
- **Tipo de variable:** Hospitalaria (identificador).
- **Descripción:** Código encriptado que identifica al médico tratante que firma formalmente el egreso del paciente.
- **Veredicto:** Se **descartó** su uso en la fase de modelado debido a su cardinalidad extremadamente alta, alcanzando hasta 17.743 valores únicos por año. Esto generaría una matriz dispersa que afectaría negativamente el desempeño de los algoritmos de selección de características, haciendo inviable su incorporación desde un punto de vista metodológico. En términos prácticos, la transformación de esta variable sería computacionalmente ineficiente y desviaría al modelo de su objetivo principal, que es identificar perfiles clínicos, demográficos y hospitalarios a nivel de red de salud.

In [15]:
# 4. Análisis de médicos de alta (MEDICOALTA_ENCRIPTADO)

# Carpeta de resultados para médicos de alta.
ruta_resultados_med_alta = "../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta"
# Crea la carpeta de salida si no existe.
os.makedirs(ruta_resultados_med_alta, exist_ok=True)
# Almacena el top 5 de médicos de alta por año.
top5_med_alta_por_año = {}
# Recorre todos los datasets anuales.
for archivo in archivos:
    # Construye la ruta completa del archivo.
    ruta = os.path.join(carpeta, archivo)
    # Extrae el año desde el nombre del archivo.
    año = archivo[-8:-4]
    # Carga el dataset correspondiente.
    df = pd.read_csv(ruta, low_memory=False)
    # Verifica la existencia del identificador del médico de alta.
    if "MEDICOALTA_ENCRIPTADO" in df.columns:
        # Convierte el identificador a texto.
        df["MEDICOALTA_ENCRIPTADO"] = (
            df["MEDICOALTA_ENCRIPTADO"].astype(str)
        )
        # Elimina registros sin información válida.
        df = df[
            ~df["MEDICOALTA_ENCRIPTADO"].isin(
                ["SIN INFORMACION", "DESCONOCIDO", "nan", "None"]
            )
        ]
        # Cuenta episodios asociados a cada médico.
        conteo_med = (
            df["MEDICOALTA_ENCRIPTADO"]
            .value_counts()
        )
        # Convierte el resultado a DataFrame.
        med_df = conteo_med.reset_index()
        # Renombra columnas descriptivamente.
        med_df.columns = [
            "MEDICOALTA_ENCRIPTADO",
            "count"
        ]
        # Define la ruta de salida.
        ruta_salida = (
            f"{ruta_resultados_med_alta}/"
            f"medicos_alta_{año}.csv"
        )
        # Exporta la distribución de médicos de alta.
        med_df.to_csv(
            ruta_salida,
            index=False,
            encoding="utf-8-sig"
        )
        # Guarda los cinco médicos con mayor volumen.
        top5_med_alta_por_año[año] = med_df.head(5)
        # Calcula el número de médicos distintos.
        num_med = med_df["MEDICOALTA_ENCRIPTADO"].nunique()
        # Calcula volumen acumulado del top 5.
        top5_volume = med_df.head(5)["count"].sum()
        # Calcula el total de episodios asociados.
        total_volume = med_df["count"].sum()
        # Calcula el porcentaje de concentración.
        concentracion = (top5_volume / total_volume) * 100
        # Reporta resultados resumidos.
        print(
            f"- {año}: {num_med} médicos de alta distintos. "
            f"Top 5 concentra {concentracion:.1f}% de episodios."
        )

- 2019: 14104 médicos de alta distintos. Top 5 concentra 14.5% de episodios.
- 2020: 16051 médicos de alta distintos. Top 5 concentra 12.4% de episodios.
- 2021: 15500 médicos de alta distintos. Top 5 concentra 11.2% de episodios.
- 2022: 15720 médicos de alta distintos. Top 5 concentra 11.2% de episodios.
- 2023: 16479 médicos de alta distintos. Top 5 concentra 8.3% de episodios.
- 2024: 17743 médicos de alta distintos. Top 5 concentra 6.5% de episodios.


In [16]:
for año in ["2019", "2020", "2021"]: # Médicos de alta más frecuentes entre 2019 y 2021
    max_hosp = top5_med_alta_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico de alta con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_alta_por_año[año].to_string(index=False))


Año 2019 (Médico de alta con mayor cantidad de intervenciones: 91808 intervenciones):
MEDICOALTA_ENCRIPTADO  count
               2932.0  91808
              16856.0  33992
                744.0  17905
              10081.0  15789
               8068.0   7099

Año 2020 (Médico de alta con mayor cantidad de intervenciones: 56847 intervenciones):
MEDICOALTA_ENCRIPTADO  count
               2945.0  56847
               2932.0  23391
              16856.0   8792
                744.0   4634
              10081.0   3101

Año 2021 (Médico de alta con mayor cantidad de intervenciones: 84036 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  84036
             70572273   1999
             68804909   1837
             73512817   1649
             73765685   1644


In [17]:
for año in ["2022", "2023", "2024"]: # Médicos de alta más frecuentes entre 2022 y 2024
    max_hosp = top5_med_alta_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico de alta con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_alta_por_año[año].to_string(index=False))


Año 2022 (Médico de alta con mayor cantidad de intervenciones: 97239 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  97239
             98438441   1890
             98195735   1778
             76891172   1696
             77772744   1521

Año 2023 (Médico de alta con mayor cantidad de intervenciones: 77100 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  77100
             98438441   3196
             71851766   1949
             73765685   1880
             68804909   1852

Año 2024 (Médico de alta con mayor cantidad de intervenciones: 62211 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434  62211
            102211195   2524
             77427459   2125
             71043147   1821
             68804909   1804


In [5]:
# Análisis global de médico de alta (2019-2024)

# Lista para almacenar todos los códigos de médico de alta 
medicos = []

# Recorre todos los datasets anuales
for archivo in archivos:
    # Construye la ruta completa del archivo
    ruta = os.path.join(carpeta, archivo)

    # Carga el dataset
    df = pd.read_csv(ruta, low_memory=False)

    # Verifica que exista la columna
    if "MEDICOALTA_ENCRIPTADO" in df.columns:
        # Agrega todos los códigos de médico de alta a la lista global
        medicos.extend(df["MEDICOALTA_ENCRIPTADO"].tolist())

# Convierte la lista a una Serie
medicos = pd.Series(medicos)

# Elimina valores faltantes (opcional, pero recomendable)
medicos = medicos.dropna()

# Cuenta episodios por médico de alta
conteo_medicos = medicos.value_counts()

# Convierte a DataFrame
conteo_medicos_df = conteo_medicos.reset_index()
conteo_medicos_df.columns = ["MEDICOALTA_ENCRIPTADO", "count"]

# Muestra estadísticas
print(f"Médicos de alta únicos: {conteo_medicos.shape[0]:,}")
print(f"Total de episodios: {conteo_medicos.sum():,}")

print("\nTop 10 médicos de alta con más episodios (2019-2024):")
print(conteo_medicos_df.head(10))

Médicos de alta únicos: 44,127
Total de episodios: 5,804,890

Top 10 médicos de alta con más episodios (2019-2024):
   MEDICOALTA_ENCRIPTADO   count
0             98438434.0  320586
1                 2932.0  115199
2                 2945.0   61989
3                16856.0   42784
4                  744.0   22539
5                10081.0   18890
6             98438441.0    7576
7                 8068.0    7100
8             68804909.0    6903
9             71851766.0    6401
